In [1]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

RAW_DIR = pathlib.Path(os.getenv("DATA_DIR_RAW", "data/raw"))
PROC_DIR = pathlib.Path(os.getenv("DATA_DIR_PROCESSED", "data/processed"))

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: /Users/that_bat/bootcamp_moshi_wang/homework/homework0/homework05/data/raw
PROC_DIR: /Users/that_bat/bootcamp_moshi_wang/homework/homework0/homework05/data/processed


In [2]:
df = pd.DataFrame({
    "date": pd.date_range("2026-08-01", periods=5),
    "ticker": ["NVDA", "AAPL", "MSFT", "AMZN", "META"],
    "price": [100.0, 110.0, 120.0, 130.0, 140.0]
})

df

,date,ticker,price
0,2026-08-01,NVDA,100.0
1,2026-08-02,AAPL,110.0
2,2026-08-03,MSFT,120.0
3,2026-08-04,AMZN,130.0
4,2026-08-05,META,140.0


In [3]:
csv_path = RAW_DIR / "sample_data.csv"
parquet_path = PROC_DIR / "sample_data.parquet"

print(csv_path)
print(parquet_path)

data/raw/sample_data.csv
data/processed/sample_data.parquet


In [4]:
df.to_csv(csv_path, index=False)
print("CSV saved:", csv_path)

CSV saved: data/raw/sample_data.csv


In [5]:
df.to_parquet(parquet_path, index=False)
print("Parquet saved:", parquet_path)

Parquet saved: data/processed/sample_data.parquet


In [9]:
df_csv = pd.read_csv(csv_path, parse_dates=["date"])
df_parquet = pd.read_parquet(parquet_path)

print("CSV shape:", df_csv.shape)
print("Parquet shape:", df_parquet.shape)

CSV shape: (5, 3)
Parquet shape: (5, 3)


In [11]:
print("original", df.dtypes)

print("\nCSV", df_csv.dtypes)

print("\nParquet", df_parquet.dtypes)

original date      datetime64[us]
ticker               str
price            float64
dtype: object

CSV date      datetime64[us]
ticker               str
price            float64
dtype: object

Parquet date      datetime64[us]
ticker               str
price            float64
dtype: object


In [13]:
def validate_df(data):
    print("shape:", data.shape)
    print("date is datetime:", pd.api.types.is_datetime64_any_dtype(data["date"]))
    print("ticker is string:", pd.api.types.is_string_dtype(data["ticker"]))
    print("price is numeric:", pd.api.types.is_numeric_dtype(data["price"]))

In [14]:
print("CSV validation:")
validate_df(df_csv)

print("\nParquet validation:")
validate_df(df_parquet)

CSV validation:
shape: (5, 3)
date is datetime: True
ticker is string: True
price is numeric: True

Parquet validation:
shape: (5, 3)
date is datetime: True
ticker is string: True
price is numeric: True


In [16]:
def write_df(data, path):
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix == ".csv":
        data.to_csv(path, index=False)

    elif path.suffix == ".parquet":
        try:
            data.to_parquet(path, index=False)
        except ImportError:
            print("parquet engine is missing. please install pyarrow or fastparquet.")

    else:
        print("unsupported file type:", path.suffix)

In [17]:
test_csv_path = RAW_DIR / "test_data.csv"
test_parquet_path = PROC_DIR / "test_data.parquet"

write_df(df, test_csv_path)
write_df(df, test_parquet_path)

print("Files written.")

Files written.


In [22]:
def read_df(path):
    path = pathlib.Path(path)

    if not path.exists():
        print("file not found:", path)
        return None

    if path.suffix == ".csv":
        return pd.read_csv(path, parse_dates=["date"])

    elif path.suffix == ".parquet":
        try:
            return pd.read_parquet(path)
        except ImportError:
            print("parquet engine is missing. Please install pyarrow or fastparquet.")
            return None

    else:
        print("unsupported file type:", path.suffix)
        return None

In [23]:
test_csv = read_df(test_csv_path)
test_parquet = read_df(test_parquet_path)

print("CSV:")
print(test_csv)

print("\nParquet:")
print(test_parquet)

CSV:
        date ticker  price
0 2026-08-01   NVDA  100.0
1 2026-08-02   AAPL  110.0
2 2026-08-03   MSFT  120.0
3 2026-08-04   AMZN  130.0
4 2026-08-05   META  140.0

Parquet:
        date ticker  price
0 2026-08-01   NVDA  100.0
1 2026-08-02   AAPL  110.0
2 2026-08-03   MSFT  120.0
3 2026-08-04   AMZN  130.0
4 2026-08-05   META  140.0


In [24]:
read_df("data/raw/does_not_exist.csv")

file not found: data/raw/does_not_exist.csv
